In [1]:
import pandas as pd
import numpy as np
import pickle

# --- PARAMS À MODIFIER ---
input_parquet = "../data/author_publications_dir/A.parquet"  # Fichier de publications
coord_csv = "../data/coordinates.csv"                       # Disciplines → coordonnées 2D

# --- Chargement des données ---
df_pub = pd.read_parquet(input_parquet)
df_coord = pd.read_csv(coord_csv, encoding="utf-8")
disciplines = df_coord.iloc[:, 0].tolist()
coord_2d = df_coord.iloc[:, 1:3].to_numpy()
disc_to_coord = {disc: xy for disc, xy in zip(disciplines, coord_2d)}

def norm(s):
    from unidecode import unidecode
    return unidecode(str(s).strip().lower())

# --- Extraction des couples (x, y) et (dx, dy) ---
all_points = []
all_dirs = []

for author, df_a in df_pub.groupby("author"):
    traj = []
    for year, df_y in df_a.groupby("year"):
        # Moyenne pondérée sur les disciplines de cette année
        dics = df_y["discipline"].value_counts()
        n = dics.sum()
        pts = [disc_to_coord[d] * count / n for d, count in dics.items() if d in disc_to_coord]
        if len(pts) == 0:
            continue
        bary = np.sum(pts, axis=0)
        traj.append((int(year), bary))
    traj.sort()
    # Ajoute les couples (point de départ, direction observée)
    for i in range(len(traj) - 1):
        year0, p0 = traj[i]
        year1, p1 = traj[i + 1]
        if year1 == year0 + 1:  # Seulement années consécutives
            all_points.append(p0)
            all_dirs.append(p1 - p0)

all_points = np.stack(all_points)
all_dirs = np.stack(all_dirs)

# --- Sauvegarde pour la suite ---
with open("points_and_dirs.pkl", "wb") as f:
    pickle.dump((all_points, all_dirs), f)
print("✅ Données extraites et sauvegardées dans points_and_dirs.pkl")


✅ Données extraites et sauvegardées dans points_and_dirs.pkl


In [2]:
import pickle
from sklearn.model_selection import train_test_split
import numpy as np

with open("points_and_dirs.pkl", "rb") as f:
    all_points, all_dirs = pickle.load(f)

# Split 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(all_points, all_dirs, test_size=0.2, random_state=42)

np.savez("train_test_data.npz", X_train=X_train, X_test=X_test, y_train=y_train, y_test=y_test)
print("✅ Données train/test sauvegardées dans train_test_data.npz")


✅ Données train/test sauvegardées dans train_test_data.npz


In [4]:
!pip install tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.0/645.0 MB 1.4 MB/s eta 0:00:00:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 1.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 507.2 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 9.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 9.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 5.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 8.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 8.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.9/71.9 kB 783.6 kB/s eta 0:00:000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.9/319.9 kB 2.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 9.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
!pip show tensorflow
import sys
print(sys.version)

Name: tensorflow
Version: 2.19.0
Summary: TensorFlow is an open source machine learning framework for everyone.
Home-page: https://www.tensorflow.org/
Author: Google Inc.
Author-email: packages@tensorflow.org
License: Apache 2.0
Location: /home/smeziou/Bureau/multdisciplinaryOnlineTool/.venv/lib/python3.12/site-packages
Requires: absl-py, astunparse, flatbuffers, gast, google-pasta, grpcio, h5py, keras, libclang, ml-dtypes, numpy, opt-einsum, packaging, protobuf, requests, setuptools, six, tensorboard, termcolor, typing-extensions, wrapt
Required-by: 
3.12.3 (main, Feb  4 2025, 14:48:35) [GCC 13.3.0]


In [ ]:
print("test")
import numpy as np
import tensorflow as tf

# Chargement train/test
print("test")
data = np.load("train_test_data.npz")
X_train = data["X_train"]
y_train = data["y_train"]
X_test = data["X_test"]
y_test = data["y_test"]
print("X_train shape:", X_train.shape, "dtype:", X_train.dtype)
print("y_train shape:", y_train.shape, "dtype:", y_train.dtype)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)
print("Exemple de X_train:", X_train[:5])
print("Exemple de y_train:", y_train[:5])
print("Nombre de points (train):", len(X_train))
print("Nombre de points (test):", len(X_test))
print("NaN dans X_train?", np.isnan(X_train).any())
print("NaN dans y_train?", np.isnan(y_train).any())
# Modèle simple MLP
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(2,)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(2)
])
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# Entraînement
print("X_train shape:", X_train.shape, "dtype:", X_train.dtype)
print("y_train shape:", y_train.shape, "dtype:", y_train.dtype)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)
print("Exemple de X_train:", X_train[:5])
print("Exemple de y_train:", y_train[:5])
print("Nombre de points (train):", len(X_train))
print("Nombre de points (test):", len(X_test))
print("NaN dans X_train?", np.isnan(X_train).any())
print("NaN dans y_train?", np.isnan(y_train).any())

model.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_test, y_test))

# Sauvegarde du modèle
model.save("direction_field_model")
print("✅ Modèle entraîné et sauvegardé dans direction_field_model/")


test
test
X_train shape: (148960, 2) dtype: float64
y_train shape: (148960, 2) dtype: float64
X_test shape: (37240, 2)
y_test shape: (37240, 2)
Exemple de X_train: [[-0.13238046 -0.13574415]
 [-0.02884499  0.10436744]
 [-0.13238046 -0.13574415]
 [-0.12589845 -0.11921068]
 [-0.13238046 -0.13574415]]
Exemple de y_train: [[ 0.          0.        ]
 [-0.00711862 -0.02976939]
 [ 0.01558554  0.02051473]
 [ 0.00037006  0.02506036]
 [ 0.20400196  0.30449473]]
Nombre de points (train): 148960
Nombre de points (test): 37240
NaN dans X_train? False
NaN dans y_train? False


NameError: name 'tf' is not defined